# 11 - Benchmark

## Objective

Membandingkan pendekatan deterministic agreement rule dan blocking strategy berdasarkan kualitas pada reviewed sample, candidate count, reduction, runtime, memory, dan kompleksitas.

## Methods compared

- `agreement_count >= 4`: high-confidence rule yang diuji pada 107 reviewed pairs.
- `agreement_count >= 2`: rule yang lebih longgar dan menghasilkan false positive pada reviewed sample.
- Blocking strategies dari tahap 07: `city_dob`, `phone_prefix_7`, `surname_prefix_dob`, `dob`, dan `email_domain`.

## Interpretation boundary

- Precision, recall, dan F1 hanya berlaku untuk 107 reviewed pairs.
- Runtime dan memory berlaku untuk snapshot dataset dan environment saat eksperimen.
- Candidate reduction bukan kualitas entity resolution.
- Benchmark tidak memilih metode berdasarkan satu metric saja.

In [ ]:
from pathlib import Path
from time import perf_counter
import tracemalloc

import pandas as pd

DATA_CANDIDATES = [
    Path.cwd() / 'data' / 'processed' / 'crm_50000_customers_standardized.csv',
    Path.cwd().parent / 'data' / 'processed' / 'crm_50000_customers_standardized.csv',
]
DATA_PATH = next((path.resolve() for path in DATA_CANDIDATES if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('Dataset terstandardisasi tidak ditemukan.')

PROCESSED_DIR = DATA_PATH.parent
REVIEW_QUEUE_PATH = PROCESSED_DIR / 'manual_review_queue.csv'
METRIC_PATH = PROCESSED_DIR / 'evaluation_metric_summary.csv'
BLOCKING_PATH = PROCESSED_DIR / 'blocking_candidate_summary.csv'
PAIR_PATH = PROCESSED_DIR / 'blocking_candidate_pairs.csv'
REQUIRED_PATHS = [REVIEW_QUEUE_PATH, METRIC_PATH, BLOCKING_PATH, PAIR_PATH]
missing_paths = [str(path) for path in REQUIRED_PATHS if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f'Artifact benchmark belum tersedia: {missing_paths}')

queue = pd.read_csv(REVIEW_QUEUE_PATH, sep=';')
metric_summary = pd.read_csv(METRIC_PATH)
blocking_summary = pd.read_csv(BLOCKING_PATH)
blocking_pairs = pd.read_csv(PAIR_PATH)

print('Dataset:', DATA_PATH)
print('Reviewed queue:', queue.shape)
print('Blocking pairs:', blocking_pairs.shape)

## Experiment 1 - Validate benchmark inputs

Benchmark hanya berjalan jika semua review queue memiliki label dan metric artifact memiliki dua rule yang diharapkan.

In [ ]:
queue['review_label'] = pd.to_numeric(queue['review_label'], errors='coerce')
queue['agreement_count'] = pd.to_numeric(queue['agreement_count'], errors='coerce')

input_checks = pd.DataFrame({
    'check': [
        'all_review_labels_available',
        'review_queue_has_107_rows',
        'metric_rules_available',
        'blocking_summary_non_empty',
        'blocking_pair_artifact_non_empty',
    ],
    'passed': [
        bool(queue['review_label'].notna().all()),
        len(queue) == 107,
        set(metric_summary['rule']) == {'agreement_count >= 4', 'agreement_count >= 2'},
        not blocking_summary.empty,
        not blocking_pairs.empty,
    ],
})
input_checks

## Experiment 2 - Reviewed-sample quality comparison

Metric dihitung dari artifact evaluasi yang sudah menggunakan label manual. Hasil ini bukan estimasi performa seluruh candidate set.

In [ ]:
rule_quality = metric_summary.copy()
rule_quality['evaluation_scope'] = '107 manually reviewed pairs'
rule_quality['complexity'] = rule_quality['rule'].map({
    'agreement_count >= 4': 'low: transparent threshold rule',
    'agreement_count >= 2': 'low: transparent threshold rule',
})
rule_quality[['rule', 'labeled_rows', 'true_positive', 'false_positive', 'false_negative', 'true_negative', 'precision', 'recall', 'f1', 'complexity']]

## Experiment 3 - Candidate and resource comparison

Candidate count dan reduction diambil dari tahap 07. Runtime dan memory adalah pengukuran pembentukan summary block pada satu eksekusi, bukan benchmark production.

In [ ]:
blocking_benchmark = blocking_summary[[
    'blocking_strategy', 'eligible_rows', 'unique_blocks',
    'repeated_blocks', 'candidate_pairs', 'largest_block',
    'candidate_reduction_percentage', 'runtime_seconds', 'peak_memory_mb',
]].copy()
blocking_benchmark['complexity'] = blocking_benchmark['blocking_strategy'].map({
    'city_dob': 'moderate: composite key, high reduction',
    'phone_prefix_7': 'moderate: prefix collision risk',
    'surname_prefix_dob': 'moderate: prefix collision risk',
    'dob': 'low: simple but broad blocks',
    'email_domain': 'low: simple but impractical candidate volume',
}).fillna('needs review')
blocking_benchmark.sort_values('candidate_pairs')

## Experiment 4 - Pair artifact build timing

Timing ini mengukur pembacaan artifact pair dan deduplication pair key. Ia memberi indikasi biaya downstream sebelum comparison/fuzzy scoring.

In [ ]:
tracemalloc.start()
start_time = perf_counter()
unique_pair_count = blocking_pairs[['left_row_index', 'right_row_index']].drop_duplicates().shape[0]
runtime_seconds = perf_counter() - start_time
_, peak_bytes = tracemalloc.get_traced_memory()
tracemalloc.stop()

pair_artifact_benchmark = pd.DataFrame({
    'metric': ['artifact_rows', 'unique_pair_rows', 'dedup_runtime_seconds', 'dedup_peak_memory_mb'],
    'value': [
        len(blocking_pairs),
        unique_pair_count,
        runtime_seconds,
        peak_bytes / (1024 ** 2),
    ],
})
pair_artifact_benchmark

## Result, Analysis, and Decision

Gunakan `rule_quality`, `blocking_benchmark`, dan `pair_artifact_benchmark` sebagai sumber hasil aktual. Jangan menyalin angka ke markdown secara manual.

### Decision framework

- Prioritaskan precision dan business risk untuk keputusan merge.
- Gunakan recall/reference coverage dan candidate reduction untuk menilai apakah blocking terlalu ketat.
- Gunakan runtime, memory, dan candidate count untuk scalability.
- Pertahankan interpretability jika dua metode memiliki kualitas yang serupa.

### Current decision boundary

Rule `agreement_count >= 4` dapat dipakai sebagai candidate high-confidence untuk review lanjutan pada sample ini. Rule tersebut belum boleh dianggap sebagai automatic merge rule untuk seluruh dataset.

`phone_prefix_7` tetap kandidat blocking yang seimbang dari tahap 07, tetapi coverage-nya bukan recall. `email_domain` tidak layak menjadi blocking tunggal karena candidate explosion.

### Limitations

- Validation hanya mencakup 107 pair.
- Review sample terstratifikasi dan tidak mewakili seluruh 70.381 comparison pairs.
- Tidak ada ground truth untuk pair di luar review queue.
- Benchmark runtime bergantung pada environment.

### Next Experiment

Gunakan hasil benchmark untuk final comparison dan dokumentasikan metode yang dipilih. Tahap berikutnya dapat berupa `12_final_comparison.ipynb`.

In [ ]:
OUTPUT_DIR = DATA_PATH.parent
RULE_OUTPUT_PATH = OUTPUT_DIR / 'benchmark_rule_quality.csv'
BLOCKING_OUTPUT_PATH = OUTPUT_DIR / 'benchmark_blocking_resources.csv'
PAIR_OUTPUT_PATH = OUTPUT_DIR / 'benchmark_pair_artifact.csv'
INPUT_CHECK_OUTPUT_PATH = OUTPUT_DIR / 'benchmark_input_checks.csv'

rule_quality.to_csv(RULE_OUTPUT_PATH, index=False)
blocking_benchmark.to_csv(BLOCKING_OUTPUT_PATH, index=False)
pair_artifact_benchmark.to_csv(PAIR_OUTPUT_PATH, index=False)
input_checks.to_csv(INPUT_CHECK_OUTPUT_PATH, index=False)

print('Saved:', RULE_OUTPUT_PATH)
print('Saved:', BLOCKING_OUTPUT_PATH)
print('Saved:', PAIR_OUTPUT_PATH)
print('Saved:', INPUT_CHECK_OUTPUT_PATH)
print('Raw dataset still exists:', (DATA_PATH.parents[1] / 'raw' / 'crm_50000_customers_dirty_v3.csv').exists())